<a href="https://colab.research.google.com/github/Ddkaba/IAD_Curs/blob/main/IAD_Cursach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.2. Переразметить исходный набор ТМИ МКА на случай бинарной
классификации: штатное функционирование, нештатное ф-е (объединить вектора с
метками отказ и сбой). Разработать приложение на языке Python с использованием
необходимых библиотек машинного и глубокого обучения на основе бинарной
классификационной модели, заданной в варианте x, которое определяет техническое
состояние малого космического аппарата: штатное ф-е 0, нештатное ф-е 1, на основе данных его ТМИ.

6.2. (для вариантов 1.2) Построить автокодировщик, определяющий аномалии – вектора, соответствующие нештатным состояниям МКА, на основании базового
нейросетевого классификатора, заданного в варианте. Обучить с валидацией на
размеченного наборе данных ТМИ, не используя эталонные метки. Протестировать
на тестовой подвыборке размеченного набора ТМИ и получить классификационные
метрики с учетом значений имеющихся эталонных метрик: сравнить результаты –
метки штатных и нештатных состояний, полученные обученным обученным
автокодировщиком, определяющим аномалии, и значения эталонных меток для
тестового поднабора размеченной выборки.  
Гибридная нейросетевая модель: последовательностное соединение  
одномерных сверточных и рекуррентных GRU нейросетевых слоев с
полносвязным классификатором: самостоятельная реализация.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import tensorflow as tf
from tensorflow.keras import layers, Model, Sequential, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import random
from deap import base, creator, tools, algorithms

In [ ]:
dataset = pd.read_csv(
    "https://raw.githubusercontent.com/Ddkaba/IAD_Curs/main/dataset.csv")
dataset.columns = [c.split(",")[0] if "," in str(c) else c for c in dataset.columns]
if "No" in dataset.columns:
    dataset = dataset.drop(columns=["No"])
# Переразметка на бинарную классификацию: штатное 0, нештатное 1 (отказ и сбой объединены)
target_col = dataset.columns[-1]
dataset[target_col] = (dataset[target_col] > 0).astype(int)

In [ ]:
TARGET = dataset.columns[-1]

print("Общая информация")
print(dataset.info())

print(f"Количество записей (объектов): {dataset.shape[0]}")
print(f"Количество признаков (фич): {dataset.shape[1]}")

print("\nНазвания столбцов:")
print(dataset.columns.tolist())

print("\nТипы данных:")
print(dataset.dtypes)

print("\nПропущенные значения:")
missing_values = dataset.isnull().sum()
print(missing_values)
print(f"Общее количество пропущенных значений: {missing_values.sum()}")

print("Целевая переменная")
if TARGET in dataset.columns:
    print(f"\nЦелевая переменная: {TARGET}")
    print(f"Тип данных целевой переменной: {dataset[TARGET].dtype}")
    unique_values = dataset[TARGET].unique()
    print(f"Уникальные значения целевой переменной (первые 20): {unique_values[:20]}")
    print(f"Всего уникальных значений: {unique_values.size}")
    if dataset[TARGET].nunique() <= 20:
        print("Распределение классов:")
        print(dataset[TARGET].value_counts())
        print("Процентное соотношение классов:")
        print(dataset[TARGET].value_counts(normalize=True) * 100)

print("Статистика")
print(dataset.describe())

In [ ]:
feat = [c for c in dataset.columns if c != TARGET]
n = len(feat)
k = 4
size = (n + k - 1) // k
chunks = [feat[i : i + size] for i in range(0, n, size)]

for i, cols in enumerate(chunks):
    block = cols + [TARGET]
    corr = dataset[block].corr(numeric_only=True)
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
    plt.title("Корреляционная матрица (признаки {}-{}, + {})".format(i * size + 1, min((i + 1) * size, n), TARGET))
    plt.tight_layout()
    plt.show()

In [ ]:
corr_with_target = dataset[feat + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET, errors="ignore")
top2 = corr_with_target.abs().nlargest(2).index.tolist()
x_col, y_col = top2[0], top2[1]

plt.figure(figsize=(10, 8))
sns.scatterplot(data=dataset, x=x_col, y=y_col, hue=TARGET, alpha=0.7)
plt.title("Диаграмма рассеивания: {} vs {} (цвет = {})".format(x_col, y_col, TARGET))
plt.tight_layout()
plt.show()

In [ ]:
feature_columns = dataset.select_dtypes(include=[np.number]).columns.tolist()
if TARGET in feature_columns:
    feature_columns.remove(TARGET)

_ = dataset[feature_columns].hist(
    bins=10,
    figsize=(20, 15),
    grid=False,
    edgecolor="black",
)
plt.suptitle("Распределение числовых признаков", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def standardize_features(dataset, target):
    """Стандартизация числовых признаков (z = (x - mean) / std), без целевой."""
    if target not in dataset.columns:
        raise ValueError(f"Целевая переменная '{target}' отсутствует в датасете.")
    base = dataset.copy()
    feature_cols = base.select_dtypes(include=[np.number]).columns.tolist()
    if target in feature_cols:
        feature_cols.remove(target)
    if not feature_cols:
        raise ValueError("Нет числовых признаков для стандартизации.")
    scaler = StandardScaler()
    scaled = scaler.fit_transform(base[feature_cols])
    standardized_df = pd.DataFrame(scaled, columns=feature_cols, index=base.index)
    out = pd.concat([standardized_df, base[[target]]], axis=1)
    print("Стандартизированы признаки:", len(feature_cols))
    print("Средние ≈ 0:", standardized_df.mean().round(4).tolist()[:5], "...")
    print("Ст. откл. ≈ 1:", standardized_df.std(ddof=0).round(4).tolist()[:5], "...")
    return out

dataset_preprocessed = standardize_features(dataset, TARGET)
feat_cols = [c for c in dataset_preprocessed.columns if c != TARGET]
_ = dataset_preprocessed[feat_cols].hist(bins=10, figsize=(20, 15), grid=False, edgecolor="black")
plt.suptitle("Распределение числовых признаков (после стандартизации)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
source_df = dataset
X_num = source_df.drop(columns=[TARGET]).select_dtypes(include=[np.number])
y = source_df[TARGET]

mi_scores = mutual_info_classif(X_num, y, random_state=0)
scores_df = (
    pd.DataFrame({"feature": X_num.columns, "score": mi_scores})
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

print("Оценки информативности (mutual information), по убыванию:")
print(scores_df)

plt.figure(figsize=(10, 12))
sns.barplot(data=scores_df, x="score", y="feature", color="#1f77b4")
plt.title("Mutual information: scores")
plt.tight_layout()
plt.show()

K = 5
selected_features = scores_df["feature"].head(K).tolist()
print(f"Топ-{K} признаков:")
print(selected_features)

In [ ]:
# П.3: Исходный и преобразованный наборы для анализа и классификации
# Исходный — все признаки; преобразованный — отбор по MI для лучшей точности

N_TOP = 30
top_features = scores_df["feature"].head(N_TOP).tolist()

dataset_orig = dataset.copy()
dataset_transformed = dataset[top_features + [TARGET]].copy()

print("Исходный набор: {} признаков".format(dataset_orig.shape[1] - 1))
print("Преобразованный набор (топ-{} по MI): {}".format(N_TOP, list(dataset_transformed.columns)))

# Стандартизация только для преобразованного набора (отбор признаков + стандартизация)
dataset_transformed_preprocessed = standardize_features(dataset_transformed, TARGET)

In [ ]:
# П.4: Проверка сбалансированности классов
counts = dataset[TARGET].value_counts()
props = dataset[TARGET].value_counts(normalize=True)
print("Распределение классов:")
print(counts)
print("\nДоли:")
print(props)
imbalance_ratio = props.max() / props.min()
if imbalance_ratio > 2.0:
    print("\nНабор несбалансирован (отношение макс/мин > 2). Оцениваем balanced_accuracy и ROC-AUC.")
else:
    print("\nНабор умеренно сбалансирован.")

In [ ]:
def fit_eval(df, name, scale=False):
    X = df.drop(columns=[TARGET])
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
    else:
        X_train, X_test = X_train.values, X_test.values
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    print("---", name, "---")
    print(classification_report(y_test, y_pred, target_names=["штатное", "нештатное"]))
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))

fit_eval(dataset_orig, "Исходный набор", scale=True)
print()
fit_eval(dataset_transformed_preprocessed, "Преобразованный набор (топ по MI + стандартизация)", scale=False)

In [ ]:
# Разбиение обоих датасетов на train / val / test (как в example: 60/20/20)
seed = 42
test_size = 0.2
val_size = 0.25  # 25% от train -> итого 60% train, 20% val, 20% test

X_orig = dataset_orig.drop(columns=[TARGET])
y = dataset_orig[TARGET]

# 1) Отделяем test
X_orig_train, X_orig_test, y_train, y_test = train_test_split(
    X_orig, y, test_size=test_size, random_state=seed, stratify=y
)
# 2) Отделяем val от train
X_orig_train, X_orig_val, y_train, y_val = train_test_split(
    X_orig_train, y_train, test_size=val_size, random_state=seed, stratify=y_train
)

# Те же индексы для преобразованного набора
train_idx = X_orig_train.index
val_idx = X_orig_val.index
test_idx = X_orig_test.index

X_trans = dataset_transformed_preprocessed.drop(columns=[TARGET])
X_trans_train = X_trans.loc[train_idx]
X_trans_val = X_trans.loc[val_idx]
X_trans_test = X_trans.loc[test_idx]

print("Исходный набор:")
print("  X_orig_train:", X_orig_train.shape, "X_orig_val:", X_orig_val.shape, "X_orig_test:", X_orig_test.shape)
print("  y_train:", y_train.shape, "y_val:", y_val.shape, "y_test:", y_test.shape)
print("Преобразованный набор:")
print("  X_trans_train:", X_trans_train.shape, "X_trans_val:", X_trans_val.shape, "X_trans_test:", X_trans_test.shape)

In [ ]:
# Гибридная модель: Conv1D + GRU + полносвязный классификатор (по обеим выборкам)

L2_REG = 1e-4
DROPOUT_RATE = 0.3

def build_hybrid_model(n_features):
    model = Sequential([
        layers.Input(shape=(n_features, 1)),
        layers.Conv1D(64, 3, activation="relu", kernel_regularizer=regularizers.L2(L2_REG)),
        layers.MaxPooling1D(2),
        layers.GRU(64, kernel_regularizer=regularizers.L2(L2_REG)),
        layers.Dropout(DROPOUT_RATE),
        layers.Dense(64, activation="relu", kernel_regularizer=regularizers.L2(L2_REG)),
        layers.Dropout(DROPOUT_RATE),
        layers.Dense(1, activation="sigmoid"),
    ], name="hybrid")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return model

def to_seq(X):
    return X.reshape(X.shape[0], X.shape[1], 1)

def train_eval_hybrid(X_train, X_val, X_test, y_train, y_val, y_test, name, scale_orig=False):
    tf.keras.backend.clear_session()
    scaler = None
    if scale_orig:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
    X_tr = to_seq(np.float32(X_train))
    X_va = to_seq(np.float32(X_val))
    X_te = to_seq(np.float32(X_test))
    y_tr = np.float32(np.asarray(y_train))
    y_va = np.float32(np.asarray(y_val))
    model = build_hybrid_model(X_tr.shape[1])
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    ]
    model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=80, batch_size=32, callbacks=callbacks, verbose=1)
    y_proba = model.predict(X_te, verbose=0).ravel()
    y_pred = (y_proba >= 0.5).astype(int)
    print("---", name, "(test) ---")
    print(classification_report(y_test, y_pred, target_names=["штатное", "нештатное"]))
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
    return model, scaler

# Обучение: исходная и преобразованная выборки
model_orig, scaler_orig = train_eval_hybrid(
    X_orig_train.values, X_orig_val.values, X_orig_test.values,
    y_train, y_val, y_test, "Исходный набор", scale_orig=True
)
print()
model_trans, _ = train_eval_hybrid(
    X_trans_train.values, X_trans_val.values, X_trans_test.values,
    y_train, y_val, y_test, "Преобразованный набор (топ по MI + стандартизация)", scale_orig=False
)

In [ ]:
# Прогон на тестовых выборках (оценка обученных моделей на test)

# Исходный набор: масштабируем test тем же scaler, предсказание
X_orig_test_scaled = scaler_orig.transform(X_orig_test.values)
X_orig_te = to_seq(np.float32(X_orig_test_scaled))
y_proba_orig = model_orig.predict(X_orig_te, verbose=0).ravel()
y_pred_orig = (y_proba_orig >= 0.5).astype(int)

print("=== Исходный набор (тестовая выборка) ===")
print(classification_report(y_test, y_pred_orig, target_names=["штатное", "нештатное"]))
print("Accuracy:", accuracy_score(y_test, y_pred_orig))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_orig))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_orig))

print()
# Преобразованный набор: test уже в том же масштабе, предсказание
X_trans_te = to_seq(np.float32(X_trans_test.values))
y_proba_trans = model_trans.predict(X_trans_te, verbose=0).ravel()
y_pred_trans = (y_proba_trans >= 0.5).astype(int)

print("=== Преобразованный набор (тестовая выборка) ===")
print(classification_report(y_test, y_pred_trans, target_names=["штатное", "нештатное"]))
print("Accuracy:", accuracy_score(y_test, y_pred_trans))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_trans))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_trans))

In [ ]:
# П.5: Поиск гиперпараметров генетическим алгоритмом (DEAP) для преобразованного набора + гибридная модель

# Данные: только преобразованная выборка (train/val)
X_tr = to_seq(np.float32(X_trans_train.values))
X_va = to_seq(np.float32(X_trans_val.values))
y_tr = np.float32(np.asarray(y_train))
y_va = np.float32(np.asarray(y_val))
n_features = X_tr.shape[1]

# Лог всех моделей в популяции (требование п.5)
ga_log = []

def build_and_train_hybrid(conv_filters, conv_kernel, gru_units, dense_units, lr, dropout, l2_reg, opt_idx, act_idx, verbose=0):
    tf.keras.backend.clear_session()
    act = "relu" if act_idx == 0 else "tanh"
    model = Sequential([
        layers.Input(shape=(n_features, 1)),
        layers.Conv1D(conv_filters, conv_kernel, activation=act, kernel_regularizer=regularizers.L2(l2_reg)),
        layers.MaxPooling1D(2),
        layers.GRU(gru_units, kernel_regularizer=regularizers.L2(l2_reg)),
        layers.Dropout(dropout),
        layers.Dense(dense_units, activation=act, kernel_regularizer=regularizers.L2(l2_reg)),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid"),
    ], name="hybrid")
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr) if opt_idx == 0 else tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ]
    model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=40, batch_size=32, callbacks=callbacks, verbose=verbose)
    y_pred = (model.predict(X_va, verbose=0).ravel() >= 0.5).astype(int)
    return balanced_accuracy_score(y_va, y_pred)

def decode(individual):
    i0, i1, i2, i3 = (max(0, min(2, int(round(individual[j])))) for j in range(4))
    conv_filters = [32, 64, 128][i0]
    conv_kernel = [2, 3, 5][i1]
    gru_units = [32, 64, 128][i2]
    dense_units = [32, 64, 128][i3]
    lr = 10 ** max(-4, min(-2, float(individual[4])))
    dropout = 0.2 + 0.3 * max(0, min(10, int(round(individual[5])))) / 10.0
    l2_reg = 10 ** max(-5, min(-3, float(individual[6])))
    opt_idx = max(0, min(1, int(round(individual[7]))))   # 0=Adam, 1=SGD
    act_idx = max(0, min(1, int(round(individual[8]))))   # 0=relu, 1=tanh
    return conv_filters, conv_kernel, gru_units, dense_units, lr, dropout, l2_reg, opt_idx, act_idx

def evaluate(individual):
    try:
        params = decode(individual)
        conv_filters, conv_kernel, gru_units, dense_units, lr, dropout, l2_reg, opt_idx, act_idx = params
        score = build_and_train_hybrid(conv_filters, conv_kernel, gru_units, dense_units, lr, dropout, l2_reg, opt_idx, act_idx, verbose=0)
        ga_log.append({"individual": list(individual), "params": params, "val_balanced_acc": score})
        return (score,)
    except Exception as e:
        ga_log.append({"individual": list(individual), "error": str(e)})
        return (0.0,)

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("idx_033", random.randint, 0, 2)   # 3 options
toolbox.register("idx_02", random.randint, 0, 2)   # kernel 2,3,5
toolbox.register("lr_log", random.uniform, -4, -2)
toolbox.register("dropout_idx", random.randint, 0, 10)
toolbox.register("l2_log", random.uniform, -5, -3)

toolbox.register("opt_idx", random.randint, 0, 1)   # 0=Adam, 1=SGD
toolbox.register("act_idx", random.randint, 0, 1)   # 0=relu, 1=tanh
toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.idx_033, toolbox.idx_02, toolbox.idx_033, toolbox.idx_033, toolbox.lr_log, toolbox.dropout_idx, toolbox.l2_log, toolbox.opt_idx, toolbox.act_idx), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

pop = toolbox.population(n=6)
NGEN = 3
for gen in range(NGEN):
    offspring = algorithms.varAnd(pop, toolbox, cxpb=0.5, mutpb=0.3)
    fits = toolbox.map(toolbox.evaluate, offspring)
    for ind, fit in zip(offspring, fits):
        ind.fitness.values = fit
    pop = toolbox.select(offspring + pop, k=len(pop))

best_ind = tools.selBest(pop, k=1)[0]
best_params = decode(best_ind)
opt_name = "Adam" if best_params[7] == 0 else "SGD"
act_name = "relu" if best_params[8] == 0 else "tanh"
print("Лучшие гиперпараметры (GA):", "conv_filters=%s conv_kernel=%s gru_units=%s dense_units=%s lr=%.2e dropout=%.2f l2=%.2e optimizer=%s activation=%s" % (best_params[0], best_params[1], best_params[2], best_params[3], best_params[4], best_params[5], best_params[6], opt_name, act_name))
print("Лог моделей в популяции (кол-во записей):", len(ga_log))

# Обучение лучшей конфигурации по GA и оценка на тесте
tf.keras.backend.clear_session()
conv_filters, conv_kernel, gru_units, dense_units, lr, dropout, l2_reg, opt_idx, act_idx = best_params
act = "relu" if act_idx == 0 else "tanh"
optimizer = tf.keras.optimizers.Adam(learning_rate=lr) if opt_idx == 0 else tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
model_ga_best = Sequential([
    layers.Input(shape=(n_features, 1)),
    layers.Conv1D(conv_filters, conv_kernel, activation=act, kernel_regularizer=regularizers.L2(l2_reg)),
    layers.MaxPooling1D(2),
    layers.GRU(gru_units, kernel_regularizer=regularizers.L2(l2_reg)),
    layers.Dropout(dropout),
    layers.Dense(dense_units, activation=act, kernel_regularizer=regularizers.L2(l2_reg)),
    layers.Dropout(dropout),
    layers.Dense(1, activation="sigmoid"),
], name="hybrid_ga_best")
model_ga_best.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
model_ga_best.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=80, batch_size=32,
    callbacks=[EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
               ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6)], verbose=1)
X_trans_te = to_seq(np.float32(X_trans_test.values))
y_proba_ga = model_ga_best.predict(X_trans_te, verbose=0).ravel()
y_pred_ga = (y_proba_ga >= 0.5).astype(int)
print("=== Лучшая модель по GA (тестовая выборка) ===")
print(classification_report(y_test, y_pred_ga, target_names=["штатное", "нештатное"]))
print("Accuracy:", accuracy_score(y_test, y_pred_ga))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_ga))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_ga))